In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import RadioButtons, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# LOW-PASS PROTOTYPE TO HIGH-PASS FREQUENCY TRANSFORMATION
#
# Exercise specifications:
#
#       ωp = 2000 rad/s
#       ωs = 1000 rad/s
#       Ap = 0.5 dB
#       As = 40 dB
#
# Prototype:
#
#       Chebyshev I
#
# Frequency transformation:
#
#       s = ω0² / p
#
# Frequency mapping:
#
#       Ω = -ω0² / ω
#
# Pole mapping:
#
#       pk = ω0² / sk
#
# Since the Chebyshev-I prototype is all-pole, N zeros appear at:
#
#       p = 0
#
# ==============================================================================
# DISPLAY STRATEGY
# ==============================================================================
#
# The prototype and transformed filters differ by several orders of magnitude.
# Plotting them simultaneously forces inappropriate axis limits and makes one
# of the two responses difficult or impossible to interpret.
#
# Therefore this notebook intentionally displays ONLY ONE FILTER AT A TIME.
#
# The user selects:
#
#       Prototype LP
#
# or:
#
#       Transformed HP
#
# using the "Filter View" radio buttons.
#
# A second radio-button group selects the displayed characteristic:
#
#       Pole-Zero Diagram
#       Magnitude Response
#       Phase Response
#       Group Delay
#       Impulse Response
#       Step Response
#
# Because only one filter is displayed at a time, every plot can use axis
# limits appropriate to the numerical scale of that particular filter.
#
# The scaling factor ω0² is NOT interactive. It is calculated once from the
# specifications of the exercise:
#
#       ω0² = Ωp ωp
#
# Since Ωp = 1:
#
#       ω0² = 2000
#
# All legends that are required are placed OUTSIDE the plotting area, on the
# right-hand side, so that they never hide poles, zeros, or response curves.
#
# No numerical result from the printed solution is hard-coded except the
# original exercise specifications. All other quantities are calculated by
# Python from the theoretical equations.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}

</style>
"""))

# ==============================================================================
# EXERCISE SPECIFICATIONS
# ==============================================================================

wp = 2000.0
ws = 1000.0
Ap = 0.5
As = 40.0

# ==============================================================================
# STEP 1: EQUIVALENT NORMALIZED LOW-PASS SPECIFICATIONS
# ==============================================================================

Omega_p = 1.0
Omega_s = wp / ws

# ==============================================================================
# STEP 2: MINIMUM CHEBYSHEV-I ORDER
# ==============================================================================

epsilon = np.sqrt(10.0**(Ap / 10.0) - 1.0)

attenuation_ratio = np.sqrt((10.0**(As / 10.0) - 1.0) / epsilon**2)

N_exact = np.arccosh(attenuation_ratio) / np.arccosh(Omega_s / Omega_p)

N = int(np.ceil(N_exact))

# ==============================================================================
# STEP 3: CHEBYSHEV-I AUXILIARY PARAMETERS
# ==============================================================================

mu = np.arcsinh(1.0 / epsilon) / N

sinh_mu = np.sinh(mu)

cosh_mu = np.cosh(mu)

# ==============================================================================
# STEP 4: PROTOTYPE CHEBYSHEV-I POLES
# ==============================================================================

k = np.arange(1, N + 1)

theta = (2.0 * k - 1.0) * np.pi / (2.0 * N)

sigma_k = -sinh_mu * np.sin(theta)

Omega_k = cosh_mu * np.cos(theta)

prototype_poles = sigma_k + 1j * Omega_k

# ==============================================================================
# STEP 5: PROTOTYPE TRANSFER FUNCTION
# ==============================================================================

prototype_den = np.real_if_close(np.poly(prototype_poles), tol=1000).real

prototype_num = np.array([prototype_den[-1]])

# ==============================================================================
# STEP 6: DESIGN SCALING VALUE
#
#       ωp = ω0² / Ωp
#
# Since Ωp = 1:
#
#       ω0² = ωp
# ==============================================================================

omega0_squared = wp * Omega_p

# ==============================================================================
# STEP 7: REQUIRED STOPBAND LIMIT FOR INTEGER ORDER
# ==============================================================================

Omega_s_required = np.cosh(np.arccosh(np.sqrt((10.0**(As / 10.0) - 1.0) / epsilon**2)) / N)

omega0_squared_stop_limit = ws * Omega_s_required

# ==============================================================================
# STEP 8: TRANSFORMED HIGH-PASS FILTER
# ==============================================================================

transformed_poles = omega0_squared / prototype_poles

transformed_zeros = np.zeros(N, dtype=complex)

transformed_den = np.real_if_close(np.poly(transformed_poles), tol=1000).real

transformed_num = np.zeros(N + 1)

transformed_num[0] = 1.0

# ==============================================================================
# FREQUENCY AXES
# ==============================================================================

Omega_axis = np.logspace(-2, 2, 5000)

omega_axis = np.logspace(1, 5, 6000)

# ==============================================================================
# PROTOTYPE FREQUENCY RESPONSE
# ==============================================================================

_, H_prototype = signal.freqs(prototype_num, prototype_den, worN=Omega_axis)

prototype_magnitude = np.abs(H_prototype)

prototype_phase = np.unwrap(np.angle(H_prototype))

prototype_phase_deg = np.rad2deg(prototype_phase)

prototype_group_delay = -np.gradient(prototype_phase, Omega_axis)

# ==============================================================================
# TRANSFORMED HIGH-PASS FREQUENCY RESPONSE
# ==============================================================================

_, H_transformed = signal.freqs(transformed_num, transformed_den, worN=omega_axis)

transformed_magnitude = np.abs(H_transformed)

transformed_phase = np.unwrap(np.angle(H_transformed))

transformed_phase_deg = np.rad2deg(transformed_phase)

transformed_group_delay = -np.gradient(transformed_phase, omega_axis)

# ==============================================================================
# TIME-DOMAIN RESPONSES
# ==============================================================================

prototype_system = signal.TransferFunction(prototype_num, prototype_den)

transformed_system = signal.TransferFunction(transformed_num, transformed_den)

# Prototype time scale
t_prototype = np.linspace(0.0, 25.0, 5000)

t_impulse_prototype, h_prototype = signal.impulse(prototype_system, T=t_prototype)

t_step_prototype, step_prototype = signal.step(prototype_system, T=t_prototype)

# ==============================================================================
# TRANSFORMED HIGH-PASS TIME SCALE
#
# The transformed high-pass filter operates on a much faster physical time
# scale. Therefore its time interval is selected from its slowest pole.
# ==============================================================================

slowest_hp_rate = np.min(np.abs(np.real(transformed_poles)))

hp_time_constant = 1.0 / slowest_hp_rate

t_highpass_max = 12.0 * hp_time_constant

t_highpass = np.linspace(0.0, t_highpass_max, 5000)

# ==============================================================================
# HIGH-PASS IMPULSE RESPONSE
#
# The high-pass transfer function has numerator and denominator of equal degree.
# Therefore it contains a direct-feedthrough term:
#
#       H_HP(p) = 1 + H_regular(p)
#
# and consequently:
#
#       h_HP(t) = δ(t) + h_regular(t)
#
# After subtracting the denominator polynomial from the numerator polynomial,
# the highest-order coefficient cancels exactly.
#
# This creates a leading zero coefficient in the numerator polynomial.
# That leading zero is removed before constructing the regular transfer
# function, because the regular part is actually of degree N-1.
#
# This avoids the scipy.signal BadCoefficients warning and is mathematically
# equivalent to using the true polynomial degree of H_regular(p).
# ==============================================================================

regular_num = transformed_num - transformed_den

regular_num = np.trim_zeros(regular_num, 'f')

regular_system = signal.TransferFunction(regular_num, transformed_den)

t_impulse_highpass, h_highpass_regular = signal.impulse(regular_system, T=t_highpass)

t_step_highpass, step_highpass = signal.step(transformed_system, T=t_highpass)

# ==============================================================================
# CHEBYSHEV POLYNOMIAL
# ==============================================================================

def chebyshev_T(N, x):

    x = np.asarray(x)

    result = np.zeros_like(x, dtype=float)

    inside = np.abs(x) <= 1.0

    outside_positive = x > 1.0

    outside_negative = x < -1.0

    result[inside] = np.cos(N * np.arccos(x[inside]))

    result[outside_positive] = np.cosh(N * np.arccosh(x[outside_positive]))

    result[outside_negative] = ((-1)**N) * np.cosh(N * np.arccosh(-x[outside_negative]))

    return result

# ==============================================================================
# CHEBYSHEV-I ATTENUATION
# ==============================================================================

def chebyshev_attenuation(Omega):

    T = chebyshev_T(N, np.asarray([Omega]))[0]

    return 10.0 * np.log10(1.0 + epsilon**2 * T**2)

# ==============================================================================
# SPECIFICATION VERIFICATION
# ==============================================================================

mapped_passband_frequency = omega0_squared / wp

mapped_stopband_frequency = omega0_squared / ws

Ap_actual = chebyshev_attenuation(mapped_passband_frequency)

As_actual = chebyshev_attenuation(mapped_stopband_frequency)

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML(f"""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:9px 11px;
    margin:0px 0px 8px 0px;
    font-size:12px;
    line-height:1.50;
    background-color:#f7fbff;
    width:1480px;
    max-width:1480px;
    box-sizing:border-box;
">

<b>Low-Pass Prototype to High-Pass Frequency Transformation</b><br>

A Chebyshev-I prototype is constructed from
ω<sub>p</sub> = {wp:.0f} rad/s,
ω<sub>s</sub> = {ws:.0f} rad/s,
A<sub>p</sub> = {Ap:.1f} dB and
A<sub>s</sub> = {As:.0f} dB.

The transformation is

<b>s = ω<sub>0</sub><sup>2</sup>/p</b>,

with

<b>ω<sub>0</sub><sup>2</sup> = {omega0_squared:.0f}</b>.

<br>

<b>Visualization strategy:</b>
The normalized prototype and the transformed physical high-pass filter are
displayed separately because their frequency, pole and time scales differ by
several orders of magnitude. Use the <b>Filter View</b> selector to examine
either filter with axis limits appropriate to its own numerical scale.

</div>
""", layout=Layout(width='1490px', max_width='1490px'))

# ==============================================================================
# FILTER-VIEW SELECTOR
# ==============================================================================

filter_view_title = HTML("""
<div style="
    font-size:13px;
    font-weight:bold;
    margin-bottom:7px;
">
Filter View
</div>
""")

filter_view_selector = RadioButtons(
    options=[
        'Prototype LP',
        'Transformed HP'
    ],
    value='Prototype LP',
    description='',
    layout=Layout(width='170px')
)

filter_view_panel = VBox(
    [filter_view_title, filter_view_selector],
    layout=Layout(
        width='205px',
        min_width='205px',
        max_width='205px',
        border='1px solid #cccccc',
        padding='9px',
        align_items='flex-start'
    )
)

# ==============================================================================
# DISPLAY SELECTOR
# ==============================================================================

display_title = HTML("""
<div style="
    font-size:13px;
    font-weight:bold;
    margin-bottom:7px;
">
Displayed Quantity
</div>
""")

display_selector = RadioButtons(
    options=[
        'Pole-Zero Diagram',
        'Magnitude Response',
        'Phase Response',
        'Group Delay',
        'Impulse Response',
        'Step Response'
    ],
    value='Pole-Zero Diagram',
    description='',
    layout=Layout(width='180px')
)

display_panel = VBox(
    [display_title, display_selector],
    layout=Layout(
        width='205px',
        min_width='205px',
        max_width='205px',
        border='1px solid #cccccc',
        padding='9px',
        align_items='flex-start'
    )
)

# ==============================================================================
# INFORMATION PANEL
# ==============================================================================

info_html = HTML(layout=Layout(width='410px', max_width='410px'))

# ==============================================================================
# POLYNOMIAL STRING
# ==============================================================================

def polynomial_string(coefficients):

    degree = len(coefficients) - 1

    terms = []

    for index, coefficient in enumerate(coefficients):

        power = degree - index

        if abs(coefficient) < 1e-8:

            continue

        if power == 0:

            terms.append(f'{coefficient:.6e}')

        elif power == 1:

            terms.append(f'{coefficient:.6e}p')

        else:

            terms.append(f'{coefficient:.6e}p^{power}')

    return ' + '.join(terms)

# ==============================================================================
# INFORMATION PANEL CONTENT
# ==============================================================================

prototype_pole_text = '<br>'.join([
    f's{k + 1} = {pole.real:+.6f} {pole.imag:+.6f}j'
    for k, pole in enumerate(prototype_poles)
])

transformed_pole_text = '<br>'.join([
    f'p{k + 1} = {pole.real:+.3f} {pole.imag:+.3f}j'
    for k, pole in enumerate(transformed_poles)
])

denominator_text = polynomial_string(transformed_den)

pass_status = '✓ satisfied' if Ap_actual <= Ap + 1e-10 else '✗ not satisfied'

stop_status = '✓ satisfied' if As_actual >= As - 1e-10 else '✗ not satisfied'

info_html.value = f"""
<div style="
    border:1px solid #cccccc;
    border-radius:7px;
    padding:9px 10px;
    font-size:11.5px;
    line-height:1.52;
    background:white;
    width:405px;
    box-sizing:border-box;
">

<b>Step 1 — Specifications</b><br>

<span style="color:#0066cc;">
ωp = {wp:.0f} rad/s,
ωs = {ws:.0f} rad/s<br>
Ap = {Ap:.1f} dB,
As = {As:.0f} dB
</span>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 2 — Equivalent prototype frequencies</b><br>

Ωp =
<span style="color:#0066cc;">{Omega_p:.6f}</span><br>

Ωs = ωp/ωs =
<span style="color:#0066cc;">{Omega_s:.6f}</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 3 — Minimum Chebyshev-I order</b><br>

Nmin =
<span style="color:#0066cc;">{N_exact:.6f}</span><br>

N =
<span style="color:#0066cc;"><b>{N}</b></span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 4 — Ripple parameter</b><br>

ε =
<span style="color:#0066cc;">{epsilon:.6f}</span><br>

μ =
<span style="color:#0066cc;">{mu:.6f}</span><br>

sinh(μ) =
<span style="color:#0066cc;">{sinh_mu:.6f}</span><br>

cosh(μ) =
<span style="color:#0066cc;">{cosh_mu:.6f}</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 5 — Prototype poles</b><br>

<span style="color:#0066cc;">
{prototype_pole_text}
</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 6 — Scaling factor</b><br>

ω₀² =
<span style="color:#0066cc;">
<b>{omega0_squared:.6f}</b>
</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 7 — High-pass poles and zeros</b><br>

<span style="color:#0066cc;">
{transformed_pole_text}
</span><br><br>

Zeros:
<span style="color:#0066cc;">
p = 0, multiplicity {N}
</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 8 — Final transfer function</b><br>

H<sub>HP</sub>(p) = p<sup>{N}</sup> / D(p)<br>

<span style="color:#0066cc;">
D(p) = {denominator_text}
</span>

</div>

<div style="margin-top:6px;padding-top:6px;border-top:1px solid #eeeeee;">

<b>Step 9 — Specification verification</b><br>

A(ωp) =
<span style="color:#0066cc;">
{Ap_actual:.6f} dB
</span>
→ {pass_status}<br>

A(ωs) =
<span style="color:#0066cc;">
{As_actual:.6f} dB
</span>
→ {stop_status}

</div>

</div>
"""

# ==============================================================================
# MAIN FIGURE
#
# One Figure and one Axes are created once.
#
# Only one filter is displayed at any given time.
#
# Space is deliberately reserved on the right side of the figure for legends.
# ==============================================================================

fig, ax = plt.subplots(figsize=(8.5, 5.5))

response_line, = ax.plot([], [], linewidth=2.3, color='red')

pole_line, = ax.plot([], [], 'ro', markersize=7)

zero_line, = ax.plot([], [], 'gx', markersize=9, markeredgewidth=1.8)

horizontal_axis = ax.axhline(0.0, color='black', linewidth=0.8)

vertical_axis = ax.axvline(0.0, color='black', linewidth=0.8)

passband_line = ax.axvline(1.0, color='black', linestyle=':', linewidth=1.0)

stopband_line = ax.axvline(2.0, color='gray', linestyle=':', linewidth=1.0)

impulse_marker, = ax.plot([], [], 'r^', markersize=9)

ax.grid(True, linestyle=':', alpha=0.5)

ax.tick_params(axis='both', labelsize=9)

# Leave space on the right for external legends
fig.subplots_adjust(left=0.12, right=0.76, bottom=0.16, top=0.88)

fig.canvas.header_visible = False

fig.canvas.toolbar_visible = False

fig.canvas.resizable = False

fig.canvas.layout.width = '850px'

fig.canvas.layout.height = '540px'

# ==============================================================================
# EXTERNAL LEGEND FUNCTION
#
# Every required legend is placed outside the plotting area, on the right.
# ==============================================================================

def external_legend(handles, labels):

    ax.legend(
        handles,
        labels,
        loc='upper left',
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        fontsize=8,
        frameon=True,
        title='Legend',
        title_fontsize=9,
        labelspacing=1.0,
        handlelength=2.5
    )

# ==============================================================================
# MAIN UPDATE FUNCTION
# ==============================================================================

def update_plot(change=None):

    view = filter_view_selector.value

    selected = display_selector.value

    # --------------------------------------------------------------------------
    # RESET VISIBILITY
    # --------------------------------------------------------------------------

    response_line.set_visible(False)

    pole_line.set_visible(False)

    zero_line.set_visible(False)

    horizontal_axis.set_visible(False)

    vertical_axis.set_visible(False)

    passband_line.set_visible(False)

    stopband_line.set_visible(False)

    impulse_marker.set_visible(False)

    old_legend = ax.get_legend()

    if old_legend is not None:

        old_legend.remove()

    # ==========================================================================
    # PROTOTYPE LOW-PASS FILTER
    # ==========================================================================

    if view == 'Prototype LP':

        response_line.set_color('blue')

        pole_line.set_color('blue')

        # ----------------------------------------------------------------------
        # POLE-ZERO DIAGRAM
        # ----------------------------------------------------------------------

        if selected == 'Pole-Zero Diagram':

            pole_line.set_visible(True)

            horizontal_axis.set_visible(True)

            vertical_axis.set_visible(True)

            pole_line.set_data(np.real(prototype_poles), np.imag(prototype_poles))

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(-1.30, 0.20)

            ax.set_ylim(-1.30, 1.30)

            ax.set_aspect('equal', adjustable='box')

            ax.set_xlabel('Re{s}', fontsize=10)

            ax.set_ylabel('Im{s}', fontsize=10)

            ax.set_title('Prototype Chebyshev-I Pole Diagram', fontsize=13, fontweight='bold', pad=8)

            external_legend([pole_line], ['Prototype poles'])

        # ----------------------------------------------------------------------
        # MAGNITUDE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Magnitude Response':

            response_line.set_visible(True)

            passband_line.set_visible(True)

            stopband_line.set_visible(True)

            response_line.set_data(Omega_axis, prototype_magnitude)

            passband_line.set_xdata([Omega_p, Omega_p])

            stopband_line.set_xdata([Omega_s, Omega_s])

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(0.01, 100.0)

            ax.set_ylim(0.0, 1.08)

            ax.set_xlabel('Normalized Angular Frequency Ω (rad/s)', fontsize=10)

            ax.set_ylabel('|Hₗₚₚ(jΩ)|', fontsize=10)

            ax.set_title('Prototype Chebyshev-I Magnitude Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # PHASE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Phase Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(Omega_axis, prototype_phase_deg)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(0.01, 100.0)

            ax.set_ylim(-470.0, 20.0)

            ax.set_xlabel('Normalized Angular Frequency Ω (rad/s)', fontsize=10)

            ax.set_ylabel('Phase (degrees)', fontsize=10)

            ax.set_title('Prototype Chebyshev-I Phase Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # GROUP DELAY
        # ----------------------------------------------------------------------

        elif selected == 'Group Delay':

            response_line.set_visible(True)

            response_line.set_data(Omega_axis, prototype_group_delay)

            gd_max = 1.10 * np.max(prototype_group_delay)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(0.01, 100.0)

            ax.set_ylim(0.0, gd_max)

            ax.set_xlabel('Normalized Angular Frequency Ω (rad/s)', fontsize=10)

            ax.set_ylabel('Group Delay', fontsize=10)

            ax.set_title('Prototype Chebyshev-I Group Delay', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # IMPULSE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Impulse Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_impulse_prototype, h_prototype)

            y_min = np.min(h_prototype)

            y_max = np.max(h_prototype)

            y_margin = 0.10 * max(y_max - y_min, 1.0)

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, 25.0)

            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_xlabel('Time t', fontsize=10)

            ax.set_ylabel('hₗₚₚ(t)', fontsize=10)

            ax.set_title('Prototype Chebyshev-I Impulse Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # STEP RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Step Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_step_prototype, step_prototype)

            y_min = min(0.0, np.min(step_prototype))

            y_max = max(1.0, np.max(step_prototype))

            y_margin = 0.08 * max(y_max - y_min, 1.0)

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, 25.0)

            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_xlabel('Time t', fontsize=10)

            ax.set_ylabel('Step Response', fontsize=10)

            ax.set_title('Prototype Chebyshev-I Step Response', fontsize=13, fontweight='bold', pad=8)

    # ==========================================================================
    # TRANSFORMED HIGH-PASS FILTER
    # ==========================================================================

    elif view == 'Transformed HP':

        response_line.set_color('red')

        pole_line.set_color('red')

        # ----------------------------------------------------------------------
        # POLE-ZERO DIAGRAM
        # ----------------------------------------------------------------------

        if selected == 'Pole-Zero Diagram':

            pole_line.set_visible(True)

            zero_line.set_visible(True)

            horizontal_axis.set_visible(True)

            vertical_axis.set_visible(True)

            pole_line.set_data(np.real(transformed_poles), np.imag(transformed_poles))

            zero_line.set_data([0.0], [0.0])

            max_real = np.max(np.abs(np.real(transformed_poles)))

            max_imag = np.max(np.abs(np.imag(transformed_poles)))

            limit = 1.15 * max(max_real, max_imag)

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(-limit, 0.15 * limit)

            ax.set_ylim(-limit, limit)

            ax.set_aspect('equal', adjustable='box')

            ax.set_xlabel('Re{p}', fontsize=10)

            ax.set_ylabel('Im{p}', fontsize=10)

            ax.set_title('Transformed High-Pass Pole-Zero Diagram', fontsize=13, fontweight='bold', pad=8)

            external_legend([pole_line, zero_line], ['High-pass poles', f'{N} zeros at p = 0'])

        # ----------------------------------------------------------------------
        # MAGNITUDE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Magnitude Response':

            response_line.set_visible(True)

            passband_line.set_visible(True)

            stopband_line.set_visible(True)

            response_line.set_data(omega_axis, transformed_magnitude)

            passband_line.set_xdata([wp, wp])

            stopband_line.set_xdata([ws, ws])

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(10.0, 1e5)

            ax.set_ylim(0.0, 1.08)

            ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

            ax.set_ylabel('|Hₕₚ(jω)|', fontsize=10)

            ax.set_title('Transformed High-Pass Magnitude Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # PHASE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Phase Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(omega_axis, transformed_phase_deg)

            phase_min = np.min(transformed_phase_deg)

            phase_max = np.max(transformed_phase_deg)

            phase_margin = 0.05 * max(phase_max - phase_min, 90.0)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(10.0, 1e5)

            ax.set_ylim(phase_min - phase_margin, phase_max + phase_margin)

            ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

            ax.set_ylabel('Phase (degrees)', fontsize=10)

            ax.set_title('Transformed High-Pass Phase Response', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # GROUP DELAY
        # ----------------------------------------------------------------------

        elif selected == 'Group Delay':

            response_line.set_visible(True)

            response_line.set_data(omega_axis, transformed_group_delay)

            finite_gd = transformed_group_delay[np.isfinite(transformed_group_delay)]

            finite_gd = finite_gd[finite_gd >= 0.0]

            gd_max = max(np.max(finite_gd), 1e-6)

            ax.set_aspect('auto')

            ax.set_xscale('log')

            ax.set_yscale('linear')

            ax.set_xlim(10.0, 1e5)

            ax.set_ylim(0.0, 1.10 * gd_max)

            ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)

            ax.set_ylabel('Group Delay (s)', fontsize=10)

            ax.set_title('Transformed High-Pass Group Delay', fontsize=13, fontweight='bold', pad=8)

        # ----------------------------------------------------------------------
        # IMPULSE RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Impulse Response':

            response_line.set_visible(True)

            impulse_marker.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_impulse_highpass, h_highpass_regular)

            y_min = np.min(h_highpass_regular)

            y_max = np.max(h_highpass_regular)

            y_abs = max(abs(y_min), abs(y_max), 1.0)

            impulse_marker.set_data([0.0], [0.85 * y_abs])

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, t_highpass_max)

            ax.set_ylim(-1.10 * y_abs, 1.10 * y_abs)

            ax.set_xlabel('Time t (s)', fontsize=10)

            ax.set_ylabel('hₕₚ(t), regular part', fontsize=10)

            ax.set_title('Transformed High-Pass Impulse Response', fontsize=13, fontweight='bold', pad=8)

            external_legend([response_line, impulse_marker], ['Regular part', 'δ(t) at t = 0'])

        # ----------------------------------------------------------------------
        # STEP RESPONSE
        # ----------------------------------------------------------------------

        elif selected == 'Step Response':

            response_line.set_visible(True)

            horizontal_axis.set_visible(True)

            response_line.set_data(t_step_highpass, step_highpass)

            y_min = min(0.0, np.min(step_highpass))

            y_max = max(1.0, np.max(step_highpass))

            y_margin = 0.08 * max(y_max - y_min, 1.0)

            ax.set_aspect('auto')

            ax.set_xscale('linear')

            ax.set_yscale('linear')

            ax.set_xlim(0.0, t_highpass_max)

            ax.set_ylim(y_min - y_margin, y_max + y_margin)

            ax.set_xlabel('Time t (s)', fontsize=10)

            ax.set_ylabel('Step Response', fontsize=10)

            ax.set_title('Transformed High-Pass Step Response', fontsize=13, fontweight='bold', pad=8)

    # --------------------------------------------------------------------------
    # ZERO-REFERENCE AXIS
    # --------------------------------------------------------------------------

    horizontal_axis.set_ydata([0.0, 0.0])

    # --------------------------------------------------------------------------
    # GRID
    # --------------------------------------------------------------------------

    ax.grid(True, which='both', linestyle=':', alpha=0.5)

    # --------------------------------------------------------------------------
    # REDRAW EXISTING FIGURE ONLY
    # --------------------------------------------------------------------------

    fig.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

filter_view_selector.observe(update_plot, names='value')

display_selector.observe(update_plot, names='value')

# ==============================================================================
# INITIALIZE
# ==============================================================================

update_plot()

# ==============================================================================
# LEFT CONTROL COLUMN
# ==============================================================================

left_column = VBox(
    [
        filter_view_panel,
        display_panel
    ],
    layout=Layout(
        width='215px',
        min_width='215px',
        max_width='215px',
        align_items='flex-start'
    )
)

# ==============================================================================
# INFORMATION COLUMN
# ==============================================================================

information_column = VBox(
    [info_html],
    layout=Layout(
        width='420px',
        min_width='420px',
        max_width='420px',
        align_items='flex-start'
    )
)

# ==============================================================================
# PLOT COLUMN
# ==============================================================================

plot_column = VBox(
    [fig.canvas],
    layout=Layout(
        width='850px',
        min_width='850px',
        max_width='850px',
        align_items='center',
        justify_content='flex-start'
    )
)

# ==============================================================================
# MAIN LAYOUT
# ==============================================================================

main_layout = HBox(
    [
        left_column,
        information_column,
        plot_column
    ],
    layout=Layout(
        width='1490px',
        align_items='flex-start',
        justify_content='flex-start'
    )
)

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_layout)